# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [1]:
%pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [2]:
from dotenv import load_dotenv 
import os 
import sys 
import json 
from pypdf import PdfReader
import gradio as gr 
from openai import AzureOpenAI 
import requests
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from agents.tracing import set_tracing_disabled
from openai.lib.azure import AsyncAzureOpenAI

/Users/brennanscomputer/Desktop/Assignment6_Gromen_Brennan/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Find the .env file in the current workspace directory
env_path = Path('.env').resolve()
load_dotenv(dotenv_path=env_path, override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")
DEPLOYMENT = "gpt-5.1-ptu"

print("Loading from:", env_path)
print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Loading from: /Users/brennanscomputer/Desktop/Assignment7_Gromen_Brennan/.env
Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [4]:
from agents import OpenAIChatCompletionsModel
from agents.tracing import set_tracing_disabled
from openai.lib.azure import AsyncAzureOpenAI

set_tracing_disabled(True)

# Clean endpoint URL to prevent duplicated /openai/ paths
raw_endpoint = AZURE_OPENAI_ENDPOINT.rstrip("/")
if raw_endpoint.endswith("/openai"):
    clean_endpoint = raw_endpoint[:-7]
else:
    clean_endpoint = raw_endpoint

azure_client = AsyncAzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21-spec",
    azure_endpoint=clean_endpoint
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client
)

## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [5]:
# Step 4: Your persona and notes
persona = """Captain Saturn Sal, a swashbuckling space pirate who sails the cosmos in search of treasure and true knowledge. He talks in a mix of pirate slang and astronomy jargon ('Ahoy, crew!', 'Shiver me starships!'), gets dramatically thrilled when you answer correctly, and groans that you've been 'lost in a black hole' when you miss one, but always explains the right answer before moving on."""

# Directly define your study notes text
notes = """
Module 7 Study Notes & Space Knowledge Base:
- Celestial Mechanics: ISS orbits Earth at roughly 28,000 km/h.
- Astronomy & Space Exploration: The International Space Station completes ~15.5 orbits per day.
- Core Course Concepts: Keep track of student missed questions and offer explanations in persona.
"""

print("Persona loaded successfully!")
print("Notes loaded length:", len(notes), "characters")

Persona loaded successfully!
Notes loaded length: 305 characters


## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [6]:
from agents import function_tool
import requests

@function_tool
def get_iss_location() -> str:
    """Get the current latitude and longitude of the International Space Station."""
    try:
        r = requests.get("http://api.open-notify.org/iss-now.json", timeout=10)
        if r.status_code != 200:
            return "The ISS tracker couldn't be reached right now (bad response). Try again later."
        pos = r.json()["iss_position"]
        return f"The ISS is currently at latitude {pos['latitude']}, longitude {pos['longitude']}."
    except requests.exceptions.RequestException:
        return "The ISS tracker is unavailable right now. Try again in a moment."

print("ISS tool registered with @function_tool successfully!")

ISS tool registered with @function_tool successfully!


## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [7]:
from agents import function_tool
import os
import json

@function_tool
def track_score_tool(correct: bool) -> str:
    """Track and update the student's correct or incorrect score in a JSON file."""
    path = "score.json"
    score = {"right": 0, "wrong": 0}
    if os.path.exists(path):
        with open(path, "r") as f:
            score = json.load(f)
    score["right" if correct else "wrong"] += 1
    with open(path, "w") as f:
        json.dump(score, f)
    return f"Score so far: {score['right']} right, {score['wrong']} wrong."

print("Step 6 score tracker tool decorated successfully!")

Step 6 score tracker tool decorated successfully!


## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [8]:
from agents import function_tool
import os
import json

@function_tool
def check_progress_tool() -> str:
    """Check the student's current accumulated score and progress."""
    path = "score.json"
    if not os.path.exists(path):
        return "No progress recorded yet. Let's start answering questions, crew!"
    with open(path, "r") as f:
        score = json.load(f)
    total = score["right"] + score["wrong"]
    if total == 0:
        return "No questions attempted yet."
    percent = (score["right"] / total) * 100
    return f"Progress report: {score['right']} correct, {score['wrong']} incorrect ({percent:.1f}% success rate)."

print("Step 7 check_progress_tool registered successfully!")

Step 7 check_progress_tool registered successfully!


## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [9]:
import os
from agents import Agent

# Combine persona, notes, and rules into instructions
instructions = f"""
{persona}

You have access to the following study notes and information base:
{notes}

Rules for using your tools:
- Use get_iss_location when the user asks about the ISS or space station coordinates.
- Use track_score_tool whenever a student answers a question to log whether they got it right or wrong.
- Use check_progress_tool whenever a student asks for their current score, progress, or stats.
"""

# Initialize the agent using azure_model defined in your setup step
your_agent = Agent(
    name="Captain Saturn Sal",
    instructions=instructions,
    tools=[get_iss_location, track_score_tool, check_progress_tool],
    model=azure_model
)

print("Agent built successfully!")

Agent built successfully!


## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [10]:
import os
from agents import Runner

# Fix the endpoint path on the existing client without touching Step 3
if hasattr(azure_client, "_base_url"):
    base_str = str(azure_client.base_url)
    if base_str.endswith("/openai/"):
        azure_client.base_url = base_str[:-8] + "/"

# Run the test
result = await Runner.run(your_agent, "Ahoy Captain! Can you tell me the current location of the ISS?")
print(result.final_output)

NotFoundError: Error code: 404 - Resource not found

## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [ ]:
import gradio as gr
from agents import Runner

async def chat(message, history):
    # Rebuild history into the message format expected by the runner/agent
    messages = []
    for human, assistant in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": assistant})
    
    # Append the current user message
    messages.append({"role": "user", "content": message})
    
    # Run your agent using the OpenAI Agents SDK Runner
    result = await Runner.run(your_agent, messages)
    return result.final_output

# Launch the Gradio chat interface in-browser
demo = gr.ChatInterface(chat)
demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/brennanscomputer/Desktop/Assignment6_Gromen_Brennan/.venv/lib/python3.14/site-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<6 lines>...
    )
    ^
  File "/Users/brennanscomputer/Desktop/Assignment6_Gromen_Brennan/.venv/lib/python3.14/site-packages/gradio/route_utils.py", line 418, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/Users/brennanscomputer/Desktop/Assignment6_Gromen_Brennan/.venv/lib/python3.14/site-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/Users/brennanscomputer/Desktop/Assignment6_Gromen_Brennan/.venv/lib/python3.14/site-packages/gradio/blocks.py", line 1961, in c

## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?
2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?
3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`?
4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts, if any, needed a real change rather than just a different decorator?

The unchanged aspects of my tool functions included things such as my ISS tracker or custom tools as well as their type hints and docstrings. These remained completely unchanged because python functions logic and JSON schema must stay identical. The changed aspects of my assignment was the execution flow. Insterad of manually handling any OpenAI tool definition schemas, I invoked a call tool and everything was replaced by the Agent(tools=[...]) command and the Runner.run() command. 

2. Compare your old call_tool(name, arguments) dispatcher to what Agent and Runner do for you now. What exactly is Runner.run doing in place of that dispatcher and your while/for loop?

In my old model, I had to manually inspect the model's message for the tool_calls command, extract the name and string arguments, and execute it manually. I also had to feed the tool output back into the message history. The new way automatically manages the agentic loop, and evaluates whether a tool call is requested.

3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into instructions?

The wording carried over directly. The behavioral instructions and guidance telling the model when it needed to choose specific tools work the same when passed into the instructions parameter of the Agent class because the model relies on the same prompt logic regardless of whether or not the loop is manual or SDK-managed. 

4. If you added a fourth tool to this agent, what would you need to write, and what would you not need to write, compared to adding a fourth tool by hand back in Module 6?

I would need to just write the python function itself with type hints, a clear docstring, and the @tool decorator then add it to my tools=[...] list. I would not need to manually update any of the JSON tools schema array, or add a new elif branch or mapping key to my old call_tool() function. I would also not need to adjust my execution loop to handle the new outputs. 

